## Setup and Imports

In [ ]:
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from utils.helpers import set_style, save_figure, start_logging
set_style()
log = start_logging(project_root, '04_01_churn_labelling')

print('All imports successful')


## Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'

# Load clean master dataset - one row per order
df_raw = pd.read_csv(processed_path / 'orders_clean_master.csv')
df_raw['PURCHASE_TS'] = pd.to_datetime(df_raw['PURCHASE_TS'], errors='coerce')
df = df_raw.copy()

# Load RFM segments from Project 02 - we use these as covariates
rfm_raw = pd.read_csv(processed_path / 'rfm_segments.csv')
rfm = rfm_raw.copy()

# Load channel ROI summary for channel covariate
orders_seg_raw = pd.read_csv(processed_path / 'orders_with_segments.csv')
orders_seg_raw['PURCHASE_TS'] = pd.to_datetime(orders_seg_raw['PURCHASE_TS'], errors='coerce')
orders_seg = orders_seg_raw.copy()

print(f'Orders loaded: {len(df):,}')
print(f'RFM customers: {len(rfm):,}')
print(f'Date range: {df["PURCHASE_TS"].min().date()} to {df["PURCHASE_TS"].max().date()}')

log(f'Orders loaded: {len(df):,}')
log(f'RFM customers: {len(rfm):,}')
log(f'Date range: {df["PURCHASE_TS"].min().date()} to {df["PURCHASE_TS"].max().date()}')


## Define the Observation Window



In [ ]:
# Observation window
# We exclude $0 price orders and null dates from the timeline
df_valid = df[(df['USD_PRICE'] > 0) & (df['PURCHASE_TS'].notna())].copy()

OBS_START = df_valid['PURCHASE_TS'].min()
OBS_END   = df_valid['PURCHASE_TS'].max()
OBS_DAYS  = (OBS_END - OBS_START).days

print(f'Observation window start: {OBS_START.date()}')
print(f'Observation window end:   {OBS_END.date()}')
print(f'Total observation days:   {OBS_DAYS}')
print(f'Valid orders in window:   {len(df_valid):,}')

log(f'Observation window: {OBS_START.date()} to {OBS_END.date()} ({OBS_DAYS} days)')
log(f'Valid orders in window:   {len(df_valid):,}')

In [ ]:
# Build customer-level timeline
# For each customer we need:
#   first_purchase: entry into observation window
#   last_purchase:  most recent purchase (used to calculate recency)
#   total_orders:   how many times they purchased
#   total_spend:    total revenue from this customer
#   tenure_days:    days from first purchase to observation end

customer_timeline = df_valid.groupby('USER_ID').agg(
    first_purchase = ('PURCHASE_TS', 'min'),
    last_purchase  = ('PURCHASE_TS', 'max'),
    total_orders   = ('ORDER_ID',    'count'),
    total_spend    = ('USD_PRICE',   'sum')
).reset_index()

# Tenure = days from first purchase to end of observation window
# This is the maximum time we could have observed this customer
customer_timeline['tenure_days'] = (
    OBS_END - customer_timeline['first_purchase']
).dt.days

# Recency = days since last purchase
customer_timeline['recency_days'] = (
    OBS_END - customer_timeline['last_purchase']
).dt.days

print(f'Customer timeline built: {len(customer_timeline):,} customers')
print()
print('Tenure distribution (days from first purchase to window end):')
print(customer_timeline['tenure_days'].describe().round(0))
print()
print('Recency distribution (days since last purchase):')
print(customer_timeline['recency_days'].describe().round(0))

log(f'Customer timeline built: {len(customer_timeline):,} customers')
log()
log('Tenure distribution (days from first purchase to window end):')
log(customer_timeline['tenure_days'].describe().round(0)) # pyright: ignore[reportArgumentType]
log()
log('Recency distribution (days since last purchase):')
log(customer_timeline['recency_days'].describe().round(0)) # pyright: ignore[reportArgumentType]


##  Data-Driven Churn Threshold


In [ ]:
# Calculate inter-purchase gaps for repeat buyers
repeat_buyers = df_valid[df_valid['USER_ID'].isin(
    customer_timeline[customer_timeline['total_orders'] > 1]['USER_ID']
)].copy()

repeat_buyers = repeat_buyers.sort_values(['USER_ID', 'PURCHASE_TS'])
repeat_buyers['prev_purchase'] = repeat_buyers.groupby('USER_ID')['PURCHASE_TS'].shift(1)
repeat_buyers['gap_days'] = (
    repeat_buyers['PURCHASE_TS'] - repeat_buyers['prev_purchase']
).dt.days

gaps = repeat_buyers['gap_days'].dropna()

print(f'Repeat buyers: {repeat_buyers["USER_ID"].nunique():,}')
print(f'Inter-purchase gaps observed: {len(gaps):,}')
print()
print('Gap distribution (days between consecutive purchases):')
print(gaps.describe().round(0))
print()

# Set churn threshold at 90th percentile of gaps with 180-day floor
p90_gap = gaps.quantile(0.90)
CHURN_THRESHOLD = max(int(p90_gap), 180)

print(f'90th percentile of inter-purchase gaps: {p90_gap:.0f} days')
print(f'Churn threshold applied:                {CHURN_THRESHOLD} days')

In [ ]:
# Apply churn label
# A customer is churned if their recency exceeds the threshold
# AND they had enough tenure to potentially churn
# (customers who only joined recently cannot have churned yet)

customer_timeline['churned'] = (
    customer_timeline['recency_days'] >= CHURN_THRESHOLD
).astype(int)

# Survival duration:
# For churned customers: days from first purchase to last purchase
# For active customers:  days from first purchase to observation end (censored)
# This is the 'T' variable in survival analysis
customer_timeline['duration'] = np.where(
    customer_timeline['churned'] == 1,
    (customer_timeline['last_purchase'] - customer_timeline['first_purchase']).dt.days,
    customer_timeline['tenure_days']
)

# Ensure minimum duration of 1 day to avoid zero-duration issues
customer_timeline['duration'] = customer_timeline['duration'].clip(lower=1)

churn_rate = customer_timeline['churned'].mean() * 100
n_churned  = customer_timeline['churned'].sum()
n_active   = len(customer_timeline) - n_churned

print(f'Churn threshold: {CHURN_THRESHOLD} days')
print(f'Churned customers: {n_churned:,} ({churn_rate:.1f}%)')
print(f'Active customers:  {n_active:,} ({100-churn_rate:.1f}%)')
print()
print('Duration statistics (days from first to last purchase or window end):')
print(customer_timeline['duration'].describe().round(0))


## Add Covariate Features


In [ ]:
# Get primary channel and region per customer
# Same approach as Notebook 3/4 in Project 02 - most frequent value
def most_frequent(x):
    counts = x.value_counts()
    return counts.index[0] if len(counts) > 0 else np.nan

customer_covariates = df_valid.groupby('USER_ID').agg(
    primary_channel = ('MARKETING_CHANNEL', most_frequent),
    primary_region  = ('REGION',            most_frequent),
    primary_product = ('PRODUCT_NAME',      most_frequent),
    avg_order_value = ('USD_PRICE',         'mean')
).reset_index()

# Merge with RFM segments
customer_covariates = customer_covariates.merge(
    rfm[['USER_ID', 'segment', 'R_score', 'F_score', 'M_score', 'RFM_score']],
    on='USER_ID', how='left'
)

print(f'Covariates built for {len(customer_covariates):,} customers')
print()
print('Channel distribution:')
print(customer_covariates['primary_channel'].value_counts())
print()
print('Region distribution:')
print(customer_covariates['primary_region'].value_counts())
print()
print('Segment distribution:')
print(customer_covariates['segment'].value_counts())

In [ ]:
# Merge timeline with covariates
survival_df = customer_timeline.merge(customer_covariates, on='USER_ID', how='left')

# Flag high-ticket buyers - customers whose primary product is a high-ticket item
high_ticket_products = [
    'Sony PlayStation 5 Bundle',
    'Lenovo IdeaPad Gaming 3',
    'Acer Nitro V Gaming Laptop'
]
survival_df['is_high_ticket_buyer'] = (
    survival_df['primary_product'].isin(high_ticket_products)
).astype(int)

# Encode channel and region as binary flags for Cox model
# Cox model needs numeric covariates - we create dummies
survival_df['is_direct']    = (survival_df['primary_channel'] == 'direct').astype(int)
survival_df['is_email']     = (survival_df['primary_channel'] == 'email').astype(int)
survival_df['is_affiliate'] = (survival_df['primary_channel'] == 'affiliate').astype(int)
survival_df['is_apac']      = (survival_df['primary_region'] == 'APAC').astype(int)
survival_df['is_emea']      = (survival_df['primary_region'] == 'EMEA').astype(int)
survival_df['is_latam']     = (survival_df['primary_region'] == 'LATAM').astype(int)

print(f'Survival dataset: {len(survival_df):,} rows, {len(survival_df.columns)} columns')
print(f'Null values per key column:')
key_cols = ['duration', 'churned', 'segment', 'primary_channel',
            'primary_region', 'RFM_score', 'avg_order_value']
for col in key_cols:
    if col in survival_df.columns:
        print(f'  {col}: {survival_df[col].isna().sum()}')

## Visualisations

In [ ]:
# Chart 1: Recency distribution with churn threshold
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(customer_timeline['recency_days'], bins=50,
        color='#2E75B6', alpha=0.7, edgecolor='white', label='All customers')
ax.axvline(CHURN_THRESHOLD, color='#C00000', linestyle='--',
           linewidth=2, label=f'Churn threshold: {CHURN_THRESHOLD} days')

churned_count = (customer_timeline['recency_days'] >= CHURN_THRESHOLD).sum()
ax.fill_betweenx(
    [0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 3000],
    CHURN_THRESHOLD, customer_timeline['recency_days'].max(),
    alpha=0.1, color='#C00000', label=f'Churned region ({churned_count:,} customers)'
)

ax.set_title('Customer recency distribution with churn threshold\n'
             f'Threshold set at {CHURN_THRESHOLD} days (90th percentile of inter-purchase gaps, min 180)',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Days since last purchase (recency)')
ax.set_ylabel('Number of customers')
ax.legend(fontsize=9)

save_figure(fig, '04_01_recency_distribution.png')
plt.show()

In [ ]:
# Chart 2: Churn rate by segment
seg_churn = survival_df.groupby('segment').agg(
    total    = ('USER_ID',  'count'),
    churned  = ('churned',  'sum')
).assign(churn_rate=lambda x: x['churned'] / x['total'] * 100)
seg_churn = seg_churn.sort_values('churn_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#C00000' if r > churn_rate else '#2E75B6'
          for r in seg_churn['churn_rate']]
bars = ax.bar(seg_churn.index, seg_churn['churn_rate'],
              color=colors, alpha=0.85)
ax.axhline(churn_rate, color='#404040', linestyle='--',
           linewidth=1.5, label=f'Overall churn rate: {churn_rate:.1f}%')

for bar, val, n in zip(bars, seg_churn['churn_rate'], seg_churn['total']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%\n(n={n:,})', ha='center', va='bottom', fontsize=9)

ax.set_title('Churn rate by RFM segment\n'
             'Red = above average churn rate',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Churn rate (%)')
ax.legend(fontsize=9)

save_figure(fig, '04_01_churn_rate_by_segment.png')
plt.show()

In [ ]:
# Chart 3: Duration distribution - churned vs active
fig, ax = plt.subplots(figsize=(12, 5))

churned_dur = survival_df[survival_df['churned'] == 1]['duration']
active_dur  = survival_df[survival_df['churned'] == 0]['duration']

ax.hist(churned_dur, bins=40, alpha=0.6, color='#C00000',
        label=f'Churned (n={len(churned_dur):,})', density=True)
ax.hist(active_dur, bins=40, alpha=0.6, color='#2E75B6',
        label=f'Active/Censored (n={len(active_dur):,})', density=True)

ax.set_title('Survival duration distribution - churned vs active customers\n'
             'Duration = days from first purchase to churn or observation end',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Duration (days)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

save_figure(fig, '04_01_duration_distribution.png')
plt.show()

In [ ]:
# Chart 4: Churn rate by channel
channel_churn = survival_df.groupby('primary_channel').agg(
    total   = ('USER_ID', 'count'),
    churned = ('churned', 'sum')
).assign(churn_rate=lambda x: x['churned'] / x['total'] * 100)
channel_churn = channel_churn[
    channel_churn['total'] >= 50
].sort_values('churn_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))

colors = ['#C00000' if r > churn_rate else '#2E75B6'
          for r in channel_churn['churn_rate']]
bars = ax.bar(channel_churn.index, channel_churn['churn_rate'],
              color=colors, alpha=0.85)
ax.axhline(churn_rate, color='#404040', linestyle='--',
           linewidth=1.5, label=f'Overall: {churn_rate:.1f}%')

for bar, val, n in zip(bars, channel_churn['churn_rate'], channel_churn['total']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%\n(n={n:,})', ha='center', va='bottom', fontsize=9)

ax.set_title('Churn rate by marketing channel',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Churn rate (%)')
ax.legend(fontsize=9)

save_figure(fig, '04_01_churn_rate_by_channel.png')
plt.show()

## Findings

In [ ]:
highest_churn_seg  = seg_churn['churn_rate'].idxmax()
lowest_churn_seg   = seg_churn['churn_rate'].idxmin()
highest_churn_ch   = channel_churn['churn_rate'].idxmax()
lowest_churn_ch    = channel_churn['churn_rate'].idxmin()

print('Notebook 1 Findings - Churn Definition and Labelling')
print()
print('Churn definition')
print(f'  Threshold: {CHURN_THRESHOLD} days since last purchase')
print(f'  Basis: 90th percentile of inter-purchase gaps among repeat buyers')
print(f'  90th percentile was 60 days - 180-day floor applied because')
print(f'  gaming hardware has naturally long repurchase cycles')
print()
print('Observation window')
print(f'  Start: {OBS_START.date()}')
print(f'  End:   {OBS_END.date()}')
print(f'  Total: {OBS_DAYS} days')
print()
print('Churn summary')
print(f'  Total customers:   {len(survival_df):,}')
print(f'  Churned:           {n_churned:,} ({churn_rate:.1f}%)')
print(f'  Active (censored): {n_active:,} ({100-churn_rate:.1f}%)')
print()
print('Finding 1 - 72.5% churn rate reflects product category, not business failure')
print('  Gaming hardware is a low-frequency, high-ticket category.')
print('  Customers buy a console or laptop once and use it for 2-4 years.')
print('  A 72.5% churn rate in a 2-year dataset is expected - most customers')
print('  have completed their purchase cycle and have no immediate reason')
print('  to return. This is structurally different from subscription or')
print('  FMCG businesses where monthly repeat purchases are the norm.')
print()
print('Finding 2 - Most churned customers have duration of 1 day')
print('  The duration distribution shows a massive spike at day 1 for')
print('  churned customers. These are one-time buyers whose first and last')
print('  purchase were the same transaction. Their survival duration is')
print('  effectively zero because they never had a second purchase.')
print('  This is the same 91% one-time buyer finding from Project 02,')
print('  now expressed through the survival analysis lens.')
print()
print('Finding 3 - Lapsed segment and churned label are perfectly aligned')
print(f'  Lapsed customers show 100% churn rate.')
print(f'  This validates both methodologies independently - the RFM')
print(f'  segmentation from Project 02 and the survival analysis churn')
print(f'  definition both identify the same customers as having left.')
print(f'  Champions show {seg_churn.loc["Champions", "churn_rate"]:.1f}% churn - the lowest of any segment,')
print(f'  confirming that higher RFM scores predict longer customer lifetimes.')
print()
print('Finding 4 - Email channel has lowest churn despite lowest CLV')
print(f'  Email churn rate: {channel_churn.loc["email", "churn_rate"]:.1f}% - lowest of all channels')
print(f'  Social media churn rate: {channel_churn.loc["social media", "churn_rate"]:.1f}% - highest')
print(f'  This inverts the Project 02 CLV finding where email ranked last.')
print(f'  Email customers spend less per transaction but stay engaged longer.')
print(f'  Social media customers buy impulsively but rarely return.')
print(f'  This distinction matters for retention strategy: email customers')
print(f'  are worth nurturing even though their individual orders are smaller.')
print()
print('Finding 5 - Inter-purchase gap data is very sparse')
print(f'  Only 1,176 inter-purchase gaps from 1,108 repeat buyers.')
print(f'  75th percentile of gaps is 0 days - meaning most repeat')
print(f'  purchases happened on the same day (likely bundle orders).')
print(f'  The 90th percentile of 60 days is driven by a small number')
print(f'  of genuine repeat buyers. The 180-day floor was therefore')
print(f'  the binding constraint, not the data-driven percentile.')
print()
print('Notebook 1 complete')
print('Figures saved to reports/figures (4 charts)')
print('Ready for Notebook 2: Kaplan-Meier Survival Curves')

log('Findings complete')
log(f'Churn rate: {churn_rate:.1f}%')
log(f'Threshold: {CHURN_THRESHOLD} days (floor binding, not percentile)')
log(f'Lapsed = 100% churn - validates RFM and survival alignment')
log(f'Email lowest churn {channel_churn.loc["email", "churn_rate"]:.1f}% despite lowest CLV')

## Export

In [ ]:
os.makedirs(processed_path, exist_ok=True)

# Save the survival-ready dataset
# This is the single input file for all subsequent notebooks
survival_output = processed_path / 'survival_data.csv'
survival_df.to_csv(survival_output, index=False)

print(f'Exported: survival_data.csv')
print(f'  Rows:    {len(survival_df):,}')
print(f'  Columns: {len(survival_df.columns)}')
print(f'  Key columns: duration, churned, segment, primary_channel,')
print(f'               primary_region, RFM_score, avg_order_value')
print()

# Save the churn threshold for use in downstream notebooks
threshold_df = pd.DataFrame([{
    'churn_threshold_days': CHURN_THRESHOLD,
    'obs_start':            str(OBS_START.date()),
    'obs_end':              str(OBS_END.date()),
    'obs_days':             OBS_DAYS,
    'n_customers':          len(survival_df),
    'n_churned':            int(n_churned),
    'churn_rate_pct':       round(churn_rate, 2)
}])
threshold_df.to_csv(processed_path / 'churn_config.csv', index=False)

print(f'Exported: churn_config.csv')
print(f'  Churn threshold: {CHURN_THRESHOLD} days')